# Reproducing the 467 MorphAgent features from images

The BBBC021 MorphAgent table used in the main figures is a **467-feature** vocabulary (`assets/morphagent_467_feature_names.csv`). Those columns were not hand-written: MorphAgent proposed them as either

- **code** features — a self-contained `extract(img, seg)` function, or
- **VLM** features — a named visual score (0–100) produced from a scoring prompt.

This notebook shows how to:

1. Pull every feature’s original code / VLM spec out of `data_test_6/results`.
2. Replay **one code feature** and **one VLM feature** on the 1-shot-per-compound subset.
3. Assemble a 467-column table on new images (full BBBC021 is documented, not re-run here).

Images live in `/data3/yez/MorphAgent/data_test_6/dataset` (3,552 fields). The smoke set is `/data3/yez/MorphAgent/data_test_6_fewshot/shot_01_per_compound/dataset` (~37 samples with `image.tif`).


## 0. Setup

Code features run locally and need no credentials. VLM features call an **OpenAI-compatible** `/v1/chat/completions` vision endpoint. **Bring your own base URL and API key** — this notebook does not ship either. Leave the prompts empty to skip VLM cells.


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

TUTORIAL = Path.cwd()
if not (TUTORIAL / "reproduce_467_features.py").exists():
    TUTORIAL = Path("/data3/yez/MorphAgent/tutorial_BBBC021")
sys.path.insert(0, str(TUTORIAL))

import importlib
import reproduce_467_features as r467
r467 = importlib.reload(r467)

RERUN_SMOKE = False
RERUN_HARVEST = False

print("Tutorial     :", TUTORIAL)
print("Library      :", r467.LIBRARY_DIR)
print("Few-shot data:", r467.FEWSHOT_DATASET)
print("Full data    :", r467.FULL_DATASET)


Tutorial     : /data3/yez/MorphAgent/tutorial_BBBC021
Library      : /data3/yez/MorphAgent/tutorial_BBBC021/feature_library
Few-shot data: /data3/yez/MorphAgent/data_test_6_fewshot/shot_01_per_compound/dataset
Full data    : /data3/yez/MorphAgent/data_test_6/dataset


### VLM credentials (required only for section 3)

Paste your OpenAI-compatible vision endpoint into the variables below (leave empty to skip VLM). Do not use `input()` / `getpass()` — the notebook would wait for a terminal prompt.

Example base URL shape: `https://<your-host>/v1`  
The endpoint must accept multimodal `image_url` messages. Model name is whatever that host serves.


In [ ]:
# Paste your endpoint here. Empty URL/key → VLM cells are skipped.
VLM_API_BASE_URL = ""   # e.g. "https://api.example.com/v1"
VLM_API_KEY = ""
VLM_MODEL = "gpt-4o"

if VLM_API_BASE_URL and VLM_API_KEY:
    r467.configure_vlm_api(VLM_API_BASE_URL, VLM_API_KEY, VLM_MODEL)
else:
    print("No VLM credentials entered. Code features still run; section 3 will skip the live API call.")


## 1. Harvest the 467 implementations

Each MorphAgent round wrote `round_*/features/<name>/extract.py` (code) or a `feature_plan.json` entry with `"method": "vlm"`. The harvester copies one canonical file per name into `feature_library/`, preferring `run_20260120_053340`.


In [ ]:
names = r467.load_467_names()
print(f"Vocabulary: {len(names)} features")
print(f"  code-like names : {sum(not n.startswith('vlm_') for n in names)}")
print(f"  vlm_ prefix     : {sum(n.startswith('vlm_') for n in names)}")

if RERUN_HARVEST or not (r467.LIBRARY_DIR / "manifest.csv").is_file():
    manifest = r467.harvest_library()
else:
    manifest = pd.read_csv(r467.LIBRARY_DIR / "manifest.csv")

display(manifest["method"].value_counts().to_frame("n"))
display(manifest.head(8))
print("...")
display(manifest[manifest["method"] == "vlm"].head(5))
assert set(manifest["method"]) <= {"code", "vlm"}
assert len(manifest) == 467


Library layout after harvest:

```
feature_library/
  manifest.csv
  code/<feature_name>/extract.py      # plus prompt/planning when present
  vlm/<feature_name>/feature.json     # name, description, category
  _shared/vlm_scoring.json            # MorphAgent scoring template
```


## 2. Replay one code feature

`tubulin_intensity_total` is a typical scalar extractor: load `image.tif` + `segmentation/*.tif`, call `extract(img, seg)`, get one float per field.


In [ ]:
code_feature = "tubulin_intensity_total"
code_path = r467.LIBRARY_DIR / "code" / code_feature / "extract.py"
print(code_path.read_text(encoding="utf-8")[:1800])


In [ ]:
fewshot_ids = r467.list_sample_ids(r467.FEWSHOT_DATASET)
print(f"Few-shot samples with image.tif: {len(fewshot_ids)}")

smoke_code_csv = r467.OUTPUT_SMOKE / f"smoke_code_{code_feature}.csv"
if RERUN_SMOKE or not smoke_code_csv.is_file():
    code_df = r467.run_code_feature(code_feature, r467.FEWSHOT_DATASET, fewshot_ids[:5])
    r467.OUTPUT_SMOKE.mkdir(parents=True, exist_ok=True)
    code_df.to_csv(smoke_code_csv, index=False)
else:
    code_df = pd.read_csv(smoke_code_csv)
display(code_df)


## 3. Replay one VLM feature

VLM features have no `extract.py`. The original planner stored a name + description; MorphAgent fills `knowledge/prompts/vlm_scoring.json` and asks a vision model for `{"score": <0-100>}`. Inputs are the three channel PNGs under `slices/`.

This cell calls **your** endpoint from section 0. If you did not enter a URL and key, it skips the live call.


In [ ]:
import json

vlm_feature = "vlm_nuclear_cap_presence"
feat = json.loads((r467.LIBRARY_DIR / "vlm" / vlm_feature / "feature.json").read_text(encoding="utf-8"))
print("name       :", feat["name"])
print("category   :", feat["category"])
print("source plan:", feat["source_plan"])
print("description:\n", feat["description"])


In [ ]:
smoke_vlm_csv = r467.OUTPUT_SMOKE / f"smoke_vlm_{vlm_feature}.csv"
r467.OUTPUT_SMOKE.mkdir(parents=True, exist_ok=True)

if not r467.vlm_api_ready():
    print("Skipping live VLM call — enter your base URL and API key in the credentials cell.")
    vlm_df = pd.read_csv(smoke_vlm_csv) if smoke_vlm_csv.is_file() else None
elif RERUN_SMOKE or not smoke_vlm_csv.is_file():
    vlm_df = r467.run_vlm_feature(vlm_feature, r467.FEWSHOT_DATASET, fewshot_ids[:2])
    vlm_df.to_csv(smoke_vlm_csv, index=False)
else:
    vlm_df = pd.read_csv(smoke_vlm_csv)
    print("Loaded cached VLM smoke CSV. Set RERUN_SMOKE = True to call your API again.")

if vlm_df is not None:
    display(vlm_df[["sample_id", vlm_feature, "error"]])


## 4. Full-dataset runtime (do not re-run here)

Smoke timings on this machine, extrapolated to **3,552** BBBC021 images:

| Workload | Rate | Serial wall time |
|----------|------|------------------|
| 439 code features | ~0.067 s / image / feature | **~29 h** |
| 28 VLM features, one API call each | ~14 s / image / feature | **~388 h** |
| Code only, 16-way process pool | — | **~2 h** |
| VLM batched (28 features, 1 call / image) | ~14 s / image | **~14 h** |

VLM dominates. A practical full rebuild is: run all **code** features in parallel, then VLM in **per-image batches** (MorphAgent’s original `vlm_scoring_batch` path). Cost and wall time depend on **your** vision API; this notebook does not launch the full run.

CLI equivalents:

```bash
python reproduce_467_features.py --harvest
python reproduce_467_features.py --smoke
```

To apply every code extractor to a dataset you would loop `r467.run_code_feature` (or `reuse_reference_run_features.py` on the original `merged_feature_code.py` rounds). Merge columns on `sample_id`, then keep the 467 names in `assets/morphagent_467_feature_names.csv`.


In [ ]:
est_path = r467.OUTPUT_SMOKE / "full_runtime_estimate.json"
if est_path.is_file():
    est = pd.read_json(est_path, typ="series")
    display(est.to_frame("value"))
else:
    print("Run the smoke cell to write the estimate JSON.")

print("\nHow to score a new image folder:")
print("  dataset_root / <sample_id> / image.tif")
print("  dataset_root / <sample_id> / segmentation/{cyto,cytoplasm,nuclei}.tif")
print("  dataset_root / <sample_id> / slices/*.png          # VLM only")


## 5. Output files

- `feature_library/manifest.csv` — 467 rows, code vs VLM, source round
- `feature_library/code/*/extract.py`
- `feature_library/vlm/*/feature.json`
- `outputs_467_smoke/smoke_code_tubulin_intensity_total.csv`
- `outputs_467_smoke/smoke_vlm_vlm_nuclear_cap_presence.csv`
- `outputs_467_smoke/full_runtime_estimate.json`
